<a href="https://colab.research.google.com/github/tejaraghavareddy/ARGUS-X-SOC/blob/main/SOC_SA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [78]:
!mkdir -p argus_sat
%cd argus_sat

print("ARGUS SAT workspace created")

/content/argus_sat/argus_sat
ARGUS SAT workspace created


In [79]:
!pip install -q streamlit pandas numpy scikit-learn plotly pytest

In [80]:
import os

folders = [
    "config",
    "ingestion",
    "analytics",
    "evidence",
    "database",
    "review",
    "synthetic_data",
    "tests",
    "docs"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project structure created")

Project structure created


In [81]:
%%writefile synthetic_data/generate.py

import pandas as pd
import numpy as np
from pathlib import Path

SEED = 42
rng = np.random.default_rng(SEED)

OUTPUT_DIR = Path("synthetic_data/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def generate_assets(n=25):
    entities = [f"ORG-{i:03d}" for i in range(1, 6)]

    rows = []

    for i in range(n):
        entity = rng.choice(entities)

        rows.append({
            "asset_id": f"AST-{i+1:04d}",
            "entity_id": entity,
            "asset_criticality": rng.choice(
                ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
            ),
            "monitoring_expected": True,
            "asset_type": rng.choice(
                ["SERVER", "DATABASE", "NETWORK", "APPLICATION"]
            )
        })

    return pd.DataFrame(rows)


def generate_cases(n=100):
    entities = [f"ORG-{i:03d}" for i in range(1, 6)]

    rows = []

    for i in range(n):

        opened = pd.Timestamp("2026-01-01") + pd.Timedelta(
            minutes=int(rng.integers(0, 60 * 24 * 90))
        )

        duration = int(rng.integers(10, 60 * 24))

        closed = opened + pd.Timedelta(minutes=duration)

        rows.append({
            "case_id": f"CASE-{i+1:05d}",
            "entity_id": rng.choice(entities),
            "severity": rng.choice(
                ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
            ),
            "status": "CLOSED",
            "opened_at": opened,
            "closed_at": closed
        })

    return pd.DataFrame(rows)


def generate_alerts(n=500):
    entities = [f"ORG-{i:03d}" for i in range(1, 6)]

    rows = []

    for i in range(n):

        opened = pd.Timestamp("2026-01-01") + pd.Timedelta(
            minutes=int(rng.integers(0, 60 * 24 * 90))
        )

        rows.append({
            "alert_id": f"ALERT-{i+1:06d}",
            "entity_id": rng.choice(entities),
            "severity": rng.choice(
                ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
            ),
            "opened_at": opened,
            "status": rng.choice(
                ["OPEN", "CLOSED"]
            )
        })

    return pd.DataFrame(rows)


def main():

    assets = generate_assets()
    cases = generate_cases()
    alerts = generate_alerts()

    assets.to_csv(OUTPUT_DIR / "assets.csv", index=False)
    cases.to_csv(OUTPUT_DIR / "cases.csv", index=False)
    alerts.to_csv(OUTPUT_DIR / "alerts.csv", index=False)

    print("Synthetic ARGUS SAT dataset generated.")
    print(f"Assets: {len(assets)}")
    print(f"Cases: {len(cases)}")
    print(f"Alerts: {len(alerts)}")


if __name__ == "__main__":
    main()

Writing synthetic_data/generate.py


In [82]:
!python synthetic_data/generate.py

Synthetic ARGUS SAT dataset generated.
Assets: 25
Cases: 100
Alerts: 500


In [83]:
import pandas as pd

assets = pd.read_csv("synthetic_data/output/assets.csv")
cases = pd.read_csv("synthetic_data/output/cases.csv")
alerts = pd.read_csv("synthetic_data/output/alerts.csv")

display(assets.head())
display(cases.head())
display(alerts.head())

,asset_id,entity_id,asset_criticality,monitoring_expected,asset_type
0,AST-0001,ORG-001,CRITICAL,True,NETWORK
1,AST-0002,ORG-003,MEDIUM,True,APPLICATION
2,AST-0003,ORG-001,HIGH,True,SERVER
3,AST-0004,ORG-001,HIGH,True,APPLICATION
4,AST-0005,ORG-004,CRITICAL,True,NETWORK


,case_id,entity_id,severity,status,opened_at,closed_at
0,CASE-00001,ORG-002,HIGH,CLOSED,2026-02-12 19:31:00,2026-02-13 03:33:00
1,CASE-00002,ORG-003,LOW,CLOSED,2026-03-02 06:47:00,2026-03-03 05:21:00
2,CASE-00003,ORG-004,LOW,CLOSED,2026-03-16 22:35:00,2026-03-17 13:45:00
3,CASE-00004,ORG-005,MEDIUM,CLOSED,2026-01-29 02:42:00,2026-01-29 21:10:00
4,CASE-00005,ORG-002,CRITICAL,CLOSED,2026-03-14 10:17:00,2026-03-15 06:30:00


,alert_id,entity_id,severity,opened_at,status
0,ALERT-000001,ORG-005,LOW,2026-03-12 15:05:00,CLOSED
1,ALERT-000002,ORG-002,CRITICAL,2026-01-10 19:45:00,OPEN
2,ALERT-000003,ORG-003,LOW,2026-03-13 17:07:00,OPEN
3,ALERT-000004,ORG-004,HIGH,2026-02-17 18:27:00,CLOSED
4,ALERT-000005,ORG-004,HIGH,2026-03-20 02:18:00,OPEN


In [84]:
%%writefile synthetic_data/generate.py

import pandas as pd
import numpy as np
from pathlib import Path

SEED = 42
rng = np.random.default_rng(SEED)

OUTPUT_DIR = Path("synthetic_data/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def generate_assets(n=25):
    entities = [f"ORG-{i:03d}" for i in range(1, 6)]

    rows = []

    for i in range(n):
        rows.append({
            "asset_id": f"AST-{i+1:04d}",
            "entity_id": rng.choice(entities),
            "asset_criticality": rng.choice(
                ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
            ),
            "monitoring_expected": True,
            "asset_type": rng.choice(
                ["SERVER", "DATABASE", "NETWORK", "APPLICATION"]
            )
        })

    return pd.DataFrame(rows)


def generate_cases(n=100):
    entities = [f"ORG-{i:03d}" for i in range(1, 6)]

    rows = []

    for i in range(n):
        opened = pd.Timestamp("2026-01-01") + pd.Timedelta(
            minutes=int(rng.integers(0, 60 * 24 * 90))
        )

        duration = int(rng.integers(10, 60 * 24))
        closed = opened + pd.Timedelta(minutes=duration)

        rows.append({
            "case_id": f"CASE-{i+1:05d}",
            "entity_id": rng.choice(entities),
            "severity": rng.choice(
                ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
            ),
            "status": "CLOSED",
            "opened_at": opened,
            "closed_at": closed
        })

    return pd.DataFrame(rows)


def generate_alerts(n=500):
    entities = [f"ORG-{i:03d}" for i in range(1, 6)]

    rows = []

    for i in range(n):
        opened = pd.Timestamp("2026-01-01") + pd.Timedelta(
            minutes=int(rng.integers(0, 60 * 24 * 90))
        )

        rows.append({
            "alert_id": f"ALERT-{i+1:06d}",
            "entity_id": rng.choice(entities),
            "severity": rng.choice(
                ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
            ),
            "opened_at": opened,
            "status": rng.choice(["OPEN", "CLOSED"])
        })

    return pd.DataFrame(rows)


def main():
    assets = generate_assets()
    cases = generate_cases()
    alerts = generate_alerts()

    assets.to_csv(OUTPUT_DIR / "assets.csv", index=False)
    cases.to_csv(OUTPUT_DIR / "cases.csv", index=False)
    alerts.to_csv(OUTPUT_DIR / "alerts.csv", index=False)

    print("Synthetic ARGUS SAT dataset generated.")
    print(f"Assets: {len(assets)}")
    print(f"Cases: {len(cases)}")
    print(f"Alerts: {len(alerts)}")


if __name__ == "__main__":
    main()

Overwriting synthetic_data/generate.py


In [85]:
!python synthetic_data/generate.py

Synthetic ARGUS SAT dataset generated.
Assets: 25
Cases: 100
Alerts: 500


In [86]:
import os

for root, dirs, files in os.walk("synthetic_data"):
    for file in files:
        print(os.path.join(root, file))


synthetic_data/generate.py
synthetic_data/output/alerts.csv
synthetic_data/output/assets.csv
synthetic_data/output/cases.csv


In [87]:
%%writefile ingestion/validate_data.py

import pandas as pd


VALID_SEVERITIES = {
    "LOW",
    "MEDIUM",
    "HIGH",
    "CRITICAL"
}


def validate_required_columns(df, required_columns):
    errors = []

    for column in required_columns:
        if column not in df.columns:
            errors.append(
                f"Missing required column: {column}"
            )

    return errors


def validate_duplicates(df, id_column):
    errors = []

    if id_column not in df.columns:
        return errors

    duplicates = df[df[id_column].duplicated(keep=False)]

    for _, row in duplicates.iterrows():
        errors.append(
            f"Duplicate {id_column}: {row[id_column]}"
        )

    return errors


def validate_severity(df):
    errors = []

    if "severity" not in df.columns:
        return errors

    invalid = df[
        ~df["severity"].astype(str).str.upper().isin(
            VALID_SEVERITIES
        )
    ]

    for index, row in invalid.iterrows():
        errors.append(
            f"Invalid severity at row {index + 2}: "
            f"{row['severity']}"
        )

    return errors


def validate_timestamps(df, timestamp_columns):
    errors = []

    for column in timestamp_columns:

        if column not in df.columns:
            continue

        parsed = pd.to_datetime(
            df[column],
            errors="coerce"
        )

        invalid = parsed.isna() & df[column].notna()

        for index in df[invalid].index:
            errors.append(
                f"Invalid timestamp in '{column}' "
                f"at row {index + 2}: "
                f"{df.loc[index, column]}"
            )

    return errors


def validate_missing_values(df, required_columns):
    warnings = []

    for column in required_columns:

        if column not in df.columns:
            continue

        missing = df[column].isna().sum()

        if missing > 0:
            warnings.append(
                f"{column}: {missing} missing values"
            )

    return warnings


def validate_dataframe(
    df,
    required_columns,
    id_column=None,
    timestamp_columns=None
):

    timestamp_columns = timestamp_columns or []

    errors = []
    warnings = []

    errors.extend(
        validate_required_columns(
            df,
            required_columns
        )
    )

    if id_column:
        errors.extend(
            validate_duplicates(
                df,
                id_column
            )
        )

    errors.extend(
        validate_severity(df)
    )

    errors.extend(
        validate_timestamps(
            df,
            timestamp_columns
        )
    )

    warnings.extend(
        validate_missing_values(
            df,
            required_columns
        )
    )

    return {
        "valid": len(errors) == 0,
        "errors": errors,
        "warnings": warnings,
        "row_count": len(df),
        "column_count": len(df.columns)
    }

Writing ingestion/validate_data.py


In [88]:
from ingestion.validate_data import validate_dataframe

result = validate_dataframe(
    cases,
    required_columns=[
        "case_id",
        "entity_id",
        "severity",
        "status",
        "opened_at",
        "closed_at"
    ],
    id_column="case_id",
    timestamp_columns=[
        "opened_at",
        "closed_at"
    ]
)

print("VALID:", result["valid"])
print("ROWS:", result["row_count"])
print("ERRORS:", len(result["errors"]))
print("WARNINGS:", len(result["warnings"]))

VALID: True
ROWS: 100
ERRORS: 0
WARNINGS: 0


In [89]:
bad_cases = cases.copy()

bad_cases.loc[0, "severity"] = "EXTREME"
bad_cases.loc[1, "opened_at"] = "NOT-A-DATE"
bad_cases.loc[2, "case_id"] = bad_cases.loc[3, "case_id"]

result = validate_dataframe(
    bad_cases,
    required_columns=[
        "case_id",
        "entity_id",
        "severity",
        "status",
        "opened_at",
        "closed_at"
    ],
    id_column="case_id",
    timestamp_columns=[
        "opened_at",
        "closed_at"
    ]
)

print("VALID:", result["valid"])
print("\nERRORS:")

for error in result["errors"]:
    print(" -", error)

VALID: False

ERRORS:
 - Duplicate case_id: CASE-00004
 - Duplicate case_id: CASE-00004
 - Invalid severity at row 2: EXTREME
 - Invalid timestamp in 'opened_at' at row 3: NOT-A-DATE


In [90]:
%%writefile ingestion/import_data.py

import json
import pandas as pd
from pathlib import Path


def load_csv(file_path):
    return pd.read_csv(file_path)


def load_json(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return pd.DataFrame(data)


def load_file(file_path):

    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(
            f"File does not exist: {file_path}"
        )

    extension = path.suffix.lower()

    if extension == ".csv":
        return load_csv(path)

    if extension == ".json":
        return load_json(path)

    raise ValueError(
        "Unsupported file type. "
        "Only CSV and JSON are supported."
    )

Writing ingestion/import_data.py


In [91]:
from ingestion.import_data import load_file

loaded_cases = load_file(
    "synthetic_data/output/cases.csv"
)

print("Successfully loaded:", len(loaded_cases))
display(loaded_cases.head())

Successfully loaded: 100


,case_id,entity_id,severity,status,opened_at,closed_at
0,CASE-00001,ORG-002,HIGH,CLOSED,2026-02-12 19:31:00,2026-02-13 03:33:00
1,CASE-00002,ORG-003,LOW,CLOSED,2026-03-02 06:47:00,2026-03-03 05:21:00
2,CASE-00003,ORG-004,LOW,CLOSED,2026-03-16 22:35:00,2026-03-17 13:45:00
3,CASE-00004,ORG-005,MEDIUM,CLOSED,2026-01-29 02:42:00,2026-01-29 21:10:00
4,CASE-00005,ORG-002,CRITICAL,CLOSED,2026-03-14 10:17:00,2026-03-15 06:30:00


In [92]:
%%writefile database/schema.py

import sqlite3
from pathlib import Path


DB_PATH = Path("argus_sat.db")


def get_connection():
    return sqlite3.connect(DB_PATH)


def initialize_database():

    conn = get_connection()
    cursor = conn.cursor()

    cursor.executescript("""

    CREATE TABLE IF NOT EXISTS organizations (
        entity_id TEXT PRIMARY KEY,
        entity_name TEXT NOT NULL,
        sector TEXT,
        peer_group TEXT
    );

    CREATE TABLE IF NOT EXISTS assets (
        asset_id TEXT PRIMARY KEY,
        entity_id TEXT NOT NULL,
        asset_criticality TEXT,
        monitoring_expected INTEGER,
        asset_type TEXT,
        source_record_id TEXT,
        submission_id TEXT
    );

    CREATE TABLE IF NOT EXISTS alerts (
        alert_id TEXT PRIMARY KEY,
        entity_id TEXT NOT NULL,
        asset_id TEXT,
        severity TEXT,
        opened_at TEXT,
        closed_at TEXT,
        status TEXT,
        source_record_id TEXT,
        submission_id TEXT
    );

    CREATE TABLE IF NOT EXISTS cases (
        case_id TEXT PRIMARY KEY,
        entity_id TEXT NOT NULL,
        severity TEXT,
        status TEXT,
        opened_at TEXT,
        closed_at TEXT,
        source_record_id TEXT,
        submission_id TEXT
    );

    CREATE TABLE IF NOT EXISTS findings (
        finding_id TEXT PRIMARY KEY,
        entity_id TEXT,
        period_start TEXT,
        period_end TEXT,
        category TEXT,
        rule_id TEXT,
        rule_version TEXT,
        severity TEXT,
        priority_score REAL,
        evidence_strength TEXT,
        explanation TEXT,
        limitations TEXT,
        status TEXT,
        created_at TEXT,
        analytics_run_id TEXT
    );

    CREATE TABLE IF NOT EXISTS finding_evidence (
        finding_id TEXT,
        source_record_type TEXT,
        source_record_id TEXT,
        source_file_id TEXT,
        source_row_or_json_pointer TEXT,
        evidence_role TEXT
    );

    CREATE TABLE IF NOT EXISTS audit_events (
        event_id TEXT PRIMARY KEY,
        actor TEXT,
        event_type TEXT,
        target_type TEXT,
        target_id TEXT,
        timestamp TEXT,
        previous_status TEXT,
        new_status TEXT,
        notes TEXT,
        analytics_run_id TEXT,
        submission_id TEXT
    );

    CREATE TABLE IF NOT EXISTS import_batches (
        submission_id TEXT PRIMARY KEY,
        entity_id TEXT,
        reporting_period_start TEXT,
        reporting_period_end TEXT,
        source_filename TEXT,
        imported_at TEXT,
        file_hash TEXT,
        accepted_records INTEGER,
        rejected_records INTEGER
    );

    """)

    conn.commit()
    conn.close()

    print("ARGUS SAT database initialized.")

Writing database/schema.py


In [93]:
from database.schema import initialize_database

initialize_database()


ARGUS SAT database initialized.


In [94]:
import sqlite3

conn = sqlite3.connect("argus_sat.db")

tables = conn.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    ORDER BY name
""").fetchall()

for table in tables:
    print(table[0])

conn.close()

alerts
assets
audit_events
cases
finding_evidence
findings
import_batches
organizations


In [95]:
%%writefile evidence/provenance.py

import hashlib
from pathlib import Path
from datetime import datetime, timezone


def calculate_file_hash(file_path):

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:
        while chunk := file.read(1024 * 1024):
            sha256.update(chunk)

    return sha256.hexdigest()


def create_submission_id(entity_id, filename):

    timestamp = datetime.now(
        timezone.utc
    ).strftime("%Y%m%d%H%M%S")

    return f"{entity_id}-{timestamp}-{Path(filename).stem}"


def create_import_metadata(
    file_path,
    entity_id,
    reporting_period_start,
    reporting_period_end
):

    path = Path(file_path)

    return {
        "submission_id": create_submission_id(
            entity_id,
            path.name
        ),
        "entity_id": entity_id,
        "source_filename": path.name,
        "reporting_period_start": reporting_period_start,
        "reporting_period_end": reporting_period_end,
        "imported_at": datetime.now(
            timezone.utc
        ).isoformat(),
        "file_hash": calculate_file_hash(path)
    }

Writing evidence/provenance.py


In [96]:
from evidence.provenance import (
    calculate_file_hash,
    create_import_metadata
)

file_path = "synthetic_data/output/cases.csv"

file_hash = calculate_file_hash(file_path)

print("SHA-256:")
print(file_hash)

SHA-256:
f0d3f9f169c848f4bcdc7510e09cb4a6b83cb6363f18c5c1c58be28ee9ac8a1c


In [97]:
metadata = create_import_metadata(
    file_path="synthetic_data/output/cases.csv",
    entity_id="ORG-001",
    reporting_period_start="2026-01-01",
    reporting_period_end="2026-03-31"
)

metadata

{'submission_id': 'ORG-001-20261030092947-cases',
 'entity_id': 'ORG-001',
 'source_filename': 'cases.csv',
 'reporting_period_start': '2026-01-01',
 'reporting_period_end': '2026-03-31',
 'imported_at': '2026-10-30T09:29:47.071547+00:00',
 'file_hash': 'f0d3f9f169c848f4bcdc7510e09cb4a6b83cb6363f18c5c1c58be28ee9ac8a1c'}

In [98]:
%%writefile database/repository.py

from database.schema import get_connection


def insert_case(
    case,
    submission_id
):

    conn = get_connection()

    conn.execute(
        """
        INSERT OR REPLACE INTO cases (
            case_id,
            entity_id,
            severity,
            status,
            opened_at,
            closed_at,
            source_record_id,
            submission_id
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            case["case_id"],
            case["entity_id"],
            case["severity"],
            case["status"],
            str(case["opened_at"]),
            str(case["closed_at"]),
            case["case_id"],
            submission_id
        )
    )

    conn.commit()
    conn.close()


def get_cases():

    conn = get_connection()

    rows = conn.execute(
        """
        SELECT *
        FROM cases
        """
    ).fetchall()

    columns = [
        "case_id",
        "entity_id",
        "severity",
        "status",
        "opened_at",
        "closed_at",
        "source_record_id",
        "submission_id"
    ]

    conn.close()

    return columns, rows

Writing database/repository.py


In [99]:
from database.repository import insert_case, get_cases

submission_id = metadata["submission_id"]

for _, case in cases.iterrows():
    insert_case(
        case,
        submission_id
    )

print("Cases inserted into SQLite.")

Cases inserted into SQLite.


In [100]:
columns, rows = get_cases()

print("Records in database:", len(rows))

for row in rows[:5]:
    print(row)

Records in database: 100
('CASE-00001', 'ORG-002', 'HIGH', 'CLOSED', '2026-02-12 19:31:00', '2026-02-13 03:33:00', 'CASE-00001', 'ORG-001-20261030092947-cases')
('CASE-00002', 'ORG-003', 'LOW', 'CLOSED', '2026-03-02 06:47:00', '2026-03-03 05:21:00', 'CASE-00002', 'ORG-001-20261030092947-cases')
('CASE-00003', 'ORG-004', 'LOW', 'CLOSED', '2026-03-16 22:35:00', '2026-03-17 13:45:00', 'CASE-00003', 'ORG-001-20261030092947-cases')
('CASE-00004', 'ORG-005', 'MEDIUM', 'CLOSED', '2026-01-29 02:42:00', '2026-01-29 21:10:00', 'CASE-00004', 'ORG-001-20261030092947-cases')
('CASE-00005', 'ORG-002', 'CRITICAL', 'CLOSED', '2026-03-14 10:17:00', '2026-03-15 06:30:00', 'CASE-00005', 'ORG-001-20261030092947-cases')


In [101]:
%%writefile analytics/rules.py

from datetime import datetime
import pandas as pd


RULE_VERSION = "1.0"


def create_finding(
    finding_id,
    entity_id,
    category,
    rule_id,
    severity,
    explanation,
    evidence,
    limitations,
    priority_score=0
):
    return {
        "finding_id": finding_id,
        "entity_id": entity_id,
        "category": category,
        "rule_id": rule_id,
        "rule_version": RULE_VERSION,
        "severity": severity,
        "priority_score": priority_score,
        "evidence_strength": "Strong",
        "explanation": explanation,
        "limitations": limitations,
        "status": "New",
        "created_at": datetime.now().isoformat(),
        "evidence": evidence
    }


# ---------------------------------------------------------
# R1 - Critical case closed unusually quickly
# ---------------------------------------------------------

def rule_r1_fast_critical_cases(cases, threshold_minutes=30):

    findings = []

    for _, case in cases.iterrows():

        if str(case["severity"]).upper() != "CRITICAL":
            continue

        opened = pd.to_datetime(
            case["opened_at"],
            errors="coerce"
        )

        closed = pd.to_datetime(
            case["closed_at"],
            errors="coerce"
        )

        if pd.isna(opened) or pd.isna(closed):
            continue

        duration = (
            closed - opened
        ).total_seconds() / 60

        if duration < threshold_minutes:

            findings.append(
                create_finding(
                    finding_id=f"R1-{case['case_id']}",
                    entity_id=case["entity_id"],
                    category="Execution Gap",
                    rule_id="R1",
                    severity="HIGH",
                    priority_score=80,
                    explanation=(
                        f"Critical case {case['case_id']} "
                        f"was closed in approximately "
                        f"{duration:.1f} minutes, below the "
                        f"configured prototype threshold of "
                        f"{threshold_minutes} minutes."
                    ),
                    evidence=[
                        {
                            "type": "case",
                            "record_id": case["case_id"],
                            "role": "supporting"
                        }
                    ],
                    limitations=(
                        "Fast closure alone does not prove "
                        "improper handling. Automated resolution "
                        "or earlier linked investigation may "
                        "explain the observation."
                    )
                )
            )

    return findings


# ---------------------------------------------------------
# R2 - Required escalation missing
# ---------------------------------------------------------

def rule_r2_missing_escalation(
    cases,
    escalations=None
):

    findings = []

    escalations = escalations or []

    escalation_case_ids = {
        e["case_id"]
        for e in escalations
    }

    for _, case in cases.iterrows():

        if str(case["severity"]).upper() not in [
            "HIGH",
            "CRITICAL"
        ]:
            continue

        case_id = case["case_id"]

        if case_id not in escalation_case_ids:

            findings.append(
                create_finding(
                    finding_id=f"R2-{case_id}",
                    entity_id=case["entity_id"],
                    category="Missing Evidence",
                    rule_id="R2",
                    severity="HIGH",
                    priority_score=75,
                    explanation=(
                        f"Case {case_id} has severity "
                        f"{case['severity']} and no corresponding "
                        f"escalation record was found in the "
                        f"submitted evidence."
                    ),
                    evidence=[
                        {
                            "type": "case",
                            "record_id": case_id,
                            "role": "supporting"
                        }
                    ],
                    limitations=(
                        "The escalation record may exist in a "
                        "system or export that was not submitted."
                    )
                )
            )

    return findings


# ---------------------------------------------------------
# R3 - Investigation evidence missing
# ---------------------------------------------------------

def rule_r3_missing_investigation(
    cases,
    investigation_actions=None
):

    findings = []

    investigation_actions = (
        investigation_actions or []
    )

    investigated_case_ids = {
        action["case_id"]
        for action in investigation_actions
    }

    for _, case in cases.iterrows():

        if str(case["status"]).upper() != "CLOSED":
            continue

        case_id = case["case_id"]

        if case_id not in investigated_case_ids:

            findings.append(
                create_finding(
                    finding_id=f"R3-{case_id}",
                    entity_id=case["entity_id"],
                    category="Missing Evidence",
                    rule_id="R3",
                    severity="HIGH",
                    priority_score=70,
                    explanation=(
                        f"Closed case {case_id} has no "
                        f"investigation action in the submitted "
                        f"evidence."
                    ),
                    evidence=[
                        {
                            "type": "case",
                            "record_id": case_id,
                            "role": "supporting"
                        }
                    ],
                    limitations=(
                        "The expected investigation workflow "
                        "must be configured for this case type. "
                        "The evidence may also be missing from "
                        "the submitted export."
                    )
                )
            )

    return findings


# ---------------------------------------------------------
# R4 - Recurring alerts without remediation
# ---------------------------------------------------------

def rule_r4_recurring_alerts(
    alerts,
    recurrence_threshold=3
):

    findings = []

    if "asset_id" not in alerts.columns:
        return findings

    grouped = (
        alerts
        .dropna(subset=["asset_id"])
        .groupby(
            ["entity_id", "asset_id"]
        )
    )

    for (entity_id, asset_id), group in grouped:

        if len(group) >= recurrence_threshold:

            alert_ids = group[
                "alert_id"
            ].tolist()

            findings.append(
                create_finding(
                    finding_id=(
                        f"R4-{entity_id}-{asset_id}"
                    ),
                    entity_id=entity_id,
                    category="Recurring Pattern",
                    rule_id="R4",
                    severity="MEDIUM",
                    priority_score=60,
                    explanation=(
                        f"Asset {asset_id} has {len(group)} "
                        f"alerts in the submitted data. "
                        f"No remediation evidence was evaluated "
                        f"by this initial prototype rule."
                    ),
                    evidence=[
                        {
                            "type": "alert",
                            "record_id": alert_id,
                            "role": "supporting"
                        }
                        for alert_id in alert_ids
                    ],
                    limitations=(
                        "Repeated alerts can be legitimate "
                        "for noisy or expected alert sources. "
                        "Remediation evidence should be checked "
                        "before treating this as a meaningful gap."
                    )
                )
            )

    return findings


# ---------------------------------------------------------
# R5 - Expected monitoring missing
# ---------------------------------------------------------

def rule_r5_missing_monitoring(
    assets,
    telemetry=None
):

    findings = []

    telemetry = telemetry or []

    monitored_assets = {
        record["asset_id"]
        for record in telemetry
    }

    for _, asset in assets.iterrows():

        expected = asset[
            "monitoring_expected"
        ]

        if not bool(expected):
            continue

        asset_id = asset["asset_id"]

        if asset_id not in monitored_assets:

            findings.append(
                create_finding(
                    finding_id=f"R5-{asset_id}",
                    entity_id=asset["entity_id"],
                    category="Negative Space",
                    rule_id="R5",
                    severity="HIGH",
                    priority_score=85,
                    explanation=(
                        f"Asset {asset_id} is explicitly marked "
                        f"as requiring monitoring, but no "
                        f"telemetry record was found in the "
                        f"submitted evidence."
                    ),
                    evidence=[
                        {
                            "type": "asset",
                            "record_id": asset_id,
                            "role": "supporting"
                        }
                    ],
                    limitations=(
                        "This finding requires sufficiently "
                        "complete asset inventory and monitoring "
                        "expectations. Absence from the submission "
                        "does not prove absence from the actual "
                        "monitoring system."
                    )
                )
            )

    return findings

Writing analytics/rules.py


In [102]:
import pandas as pd

test_cases = cases.copy()

test_cases.loc[0, "severity"] = "CRITICAL"
test_cases.loc[0, "opened_at"] = "2026-02-01 10:00:00"
test_cases.loc[0, "closed_at"] = "2026-02-01 10:10:00"

from analytics.rules import rule_r1_fast_critical_cases

r1_findings = rule_r1_fast_critical_cases(
    test_cases,
    threshold_minutes=30
)

print("R1 findings:", len(r1_findings))

for finding in r1_findings[:3]:
    print()
    print(finding["finding_id"])
    print(finding["explanation"])

R1 findings: 2

R1-CASE-00001
Critical case CASE-00001 was closed in approximately 10.0 minutes, below the configured prototype threshold of 30 minutes.

R1-CASE-00077
Critical case CASE-00077 was closed in approximately 17.0 minutes, below the configured prototype threshold of 30 minutes.


In [103]:
from analytics.rules import rule_r3_missing_investigation

r3_findings = rule_r3_missing_investigation(
    test_cases,
    investigation_actions=[]
)

print("R3 findings:", len(r3_findings))

for finding in r3_findings[:3]:
    print(finding["finding_id"])

R3 findings: 100
R3-CASE-00001
R3-CASE-00002
R3-CASE-00003


In [104]:
from analytics.rules import rule_r5_missing_monitoring

r5_findings = rule_r5_missing_monitoring(
    assets,
    telemetry=[]
)

print("R5 findings:", len(r5_findings))

for finding in r5_findings[:3]:
    print(finding["finding_id"])
    print(finding["explanation"])
    print()

R5 findings: 25
R5-AST-0001
Asset AST-0001 is explicitly marked as requiring monitoring, but no telemetry record was found in the submitted evidence.

R5-AST-0002
Asset AST-0002 is explicitly marked as requiring monitoring, but no telemetry record was found in the submitted evidence.

R5-AST-0003
Asset AST-0003 is explicitly marked as requiring monitoring, but no telemetry record was found in the submitted evidence.



In [106]:
%%writefile analytics/run_rules.py

from analytics.rules import (
    rule_r1_fast_critical_cases,
    rule_r2_missing_escalation,
    rule_r3_missing_investigation,
    rule_r4_recurring_alerts,
    rule_r5_missing_monitoring
)


def run_all_rules(
    cases,
    alerts,
    assets,
    escalations=None,
    investigation_actions=None,
    telemetry=None
):

    findings = []

    findings.extend(
        rule_r1_fast_critical_cases(cases)
    )

    findings.extend(
        rule_r2_missing_escalation(
            cases,
            escalations
        )
    )

    findings.extend(
        rule_r3_missing_investigation(
            cases,
            investigation_actions
        )
    )

    findings.extend(
        rule_r4_recurring_alerts(alerts)
    )

    findings.extend(
        rule_r5_missing_monitoring(
            assets,
            telemetry
        )
    )

    return findings

Overwriting analytics/run_rules.py


In [107]:
from analytics.run_rules import run_all_rules

findings = run_all_rules(
    cases=test_cases,
    alerts=alerts,
    assets=assets,
    escalations=[],
    investigation_actions=[],
    telemetry=[]
)

print("TOTAL FINDINGS:", len(findings))

for finding in findings[:10]:
    print(
        finding["finding_id"],
        "|",
        finding["rule_id"],
        "|",
        finding["severity"]
    )

TOTAL FINDINGS: 171
R1-CASE-00001 | R1 | HIGH
R1-CASE-00077 | R1 | HIGH
R2-CASE-00001 | R2 | HIGH
R2-CASE-00005 | R2 | HIGH
R2-CASE-00006 | R2 | HIGH
R2-CASE-00007 | R2 | HIGH
R2-CASE-00008 | R2 | HIGH
R2-CASE-00010 | R2 | HIGH
R2-CASE-00012 | R2 | HIGH
R2-CASE-00014 | R2 | HIGH


In [108]:
%%writefile database/findings_repository.py

import json
from database.schema import get_connection


def save_findings(findings, analytics_run_id):

    conn = get_connection()

    for finding in findings:

        conn.execute(
            """
            INSERT OR REPLACE INTO findings (
                finding_id,
                entity_id,
                period_start,
                period_end,
                category,
                rule_id,
                rule_version,
                severity,
                priority_score,
                evidence_strength,
                explanation,
                limitations,
                status,
                created_at,
                analytics_run_id
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (
                finding["finding_id"],
                finding["entity_id"],
                None,
                None,
                finding["category"],
                finding["rule_id"],
                finding["rule_version"],
                finding["severity"],
                finding["priority_score"],
                finding["evidence_strength"],
                finding["explanation"],
                finding["limitations"],
                finding["status"],
                finding["created_at"],
                analytics_run_id
            )
        )

        # Save evidence links
        for evidence in finding.get("evidence", []):

            conn.execute(
                """
                INSERT INTO finding_evidence (
                    finding_id,
                    source_record_type,
                    source_record_id,
                    source_file_id,
                    source_row_or_json_pointer,
                    evidence_role
                )
                VALUES (?, ?, ?, ?, ?, ?)
                """,
                (
                    finding["finding_id"],
                    evidence.get("type"),
                    evidence.get("record_id"),
                    evidence.get("source_file_id"),
                    evidence.get("source_row_or_json_pointer"),
                    evidence.get("role")
                )
            )

    conn.commit()
    conn.close()


def get_findings():

    conn = get_connection()

    rows = conn.execute(
        """
        SELECT
            finding_id,
            entity_id,
            category,
            rule_id,
            rule_version,
            severity,
            priority_score,
            evidence_strength,
            explanation,
            limitations,
            status,
            created_at,
            analytics_run_id
        FROM findings
        ORDER BY priority_score DESC
        """
    ).fetchall()

    columns = [
        "finding_id",
        "entity_id",
        "category",
        "rule_id",
        "rule_version",
        "severity",
        "priority_score",
        "evidence_strength",
        "explanation",
        "limitations",
        "status",
        "created_at",
        "analytics_run_id"
    ]

    conn.close()

    return columns, rows

Writing database/findings_repository.py


In [109]:
from database.findings_repository import save_findings

analytics_run_id = "RUN-001"

save_findings(
    findings,
    analytics_run_id
)

print("Findings saved successfully.")

Findings saved successfully.


In [110]:
from database.findings_repository import get_findings

columns, rows = get_findings()

print("Total stored findings:", len(rows))

for row in rows[:10]:
    print(row[:8])

Total stored findings: 171
('R5-AST-0001', 'ORG-001', 'Negative Space', 'R5', '1.0', 'HIGH', 85.0, 'Strong')
('R5-AST-0002', 'ORG-003', 'Negative Space', 'R5', '1.0', 'HIGH', 85.0, 'Strong')
('R5-AST-0003', 'ORG-001', 'Negative Space', 'R5', '1.0', 'HIGH', 85.0, 'Strong')
('R5-AST-0004', 'ORG-001', 'Negative Space', 'R5', '1.0', 'HIGH', 85.0, 'Strong')
('R5-AST-0005', 'ORG-004', 'Negative Space', 'R5', '1.0', 'HIGH', 85.0, 'Strong')
('R5-AST-0006', 'ORG-004', 'Negative Space', 'R5', '1.0', 'HIGH', 85.0, 'Strong')
('R5-AST-0007', 'ORG-005', 'Negative Space', 'R5', '1.0', 'HIGH', 85.0, 'Strong')
('R5-AST-0008', 'ORG-002', 'Negative Space', 'R5', '1.0', 'HIGH', 85.0, 'Strong')
('R5-AST-0009', 'ORG-004', 'Negative Space', 'R5', '1.0', 'HIGH', 85.0, 'Strong')
('R5-AST-0010', 'ORG-005', 'Negative Space', 'R5', '1.0', 'HIGH', 85.0, 'Strong')


In [111]:
%%writefile analytics/negative_space.py

import pandas as pd


def find_missing_expected_records(
    expected_df,
    actual_df,
    key_column,
    expected_condition=None
):
    """
    Find records expected to exist but absent
    from the submitted evidence.
    """

    expected = expected_df.copy()

    if expected_condition is not None:
        expected = expected[
            expected_condition(expected)
        ]

    expected_ids = set(
        expected[key_column].dropna()
    )

    actual_ids = set()

    if actual_df is not None and key_column in actual_df.columns:
        actual_ids = set(
            actual_df[key_column].dropna()
        )

    missing_ids = expected_ids - actual_ids

    return expected[
        expected[key_column].isin(missing_ids)
    ].copy()

Writing analytics/negative_space.py


In [112]:
from analytics.negative_space import find_missing_expected_records

missing_assets = find_missing_expected_records(
    expected_df=assets,
    actual_df=pd.DataFrame(
        columns=["asset_id"]
    ),
    key_column="asset_id",
    expected_condition=lambda df:
        df["monitoring_expected"] == True
)

print(
    "Expected assets without telemetry:",
    len(missing_assets)
)

display(
    missing_assets[
        ["asset_id", "entity_id", "asset_type"]
    ].head(10)
)

Expected assets without telemetry: 25


,asset_id,entity_id,asset_type
0,AST-0001,ORG-001,NETWORK
1,AST-0002,ORG-003,APPLICATION
2,AST-0003,ORG-001,SERVER
3,AST-0004,ORG-001,APPLICATION
4,AST-0005,ORG-004,NETWORK
5,AST-0006,ORG-004,SERVER
6,AST-0007,ORG-005,NETWORK
7,AST-0008,ORG-002,APPLICATION
8,AST-0009,ORG-004,DATABASE
9,AST-0010,ORG-005,DATABASE


In [113]:
%%writefile analytics/prioritization.py

SEVERITY_WEIGHT = {
    "LOW": 10,
    "MEDIUM": 30,
    "HIGH": 60,
    "CRITICAL": 80
}

EVIDENCE_WEIGHT = {
    "Weak": 5,
    "Moderate": 10,
    "Strong": 20
}


def calculate_priority(
    severity,
    evidence_strength,
    recurrence=0,
    affected_scope=1
):

    severity_score = SEVERITY_WEIGHT.get(
        str(severity).upper(),
        0
    )

    evidence_score = EVIDENCE_WEIGHT.get(
        evidence_strength,
        0
    )

    recurrence_score = min(
        recurrence * 2,
        20
    )

    scope_score = min(
        affected_scope * 2,
        20
    )

    total = (
        severity_score
        + evidence_score
        + recurrence_score
        + scope_score
    )

    return min(total, 100)

Writing analytics/prioritization.py


In [114]:
%%writefile app.py

import streamlit as st
import pandas as pd

from database.schema import initialize_database
from database.findings_repository import get_findings


# ---------------------------------------------------------
# Page configuration
# ---------------------------------------------------------

st.set_page_config(
    page_title="ARGUS SAT",
    page_icon="🔎",
    layout="wide"
)


# ---------------------------------------------------------
# Initialize local database
# ---------------------------------------------------------

initialize_database()


# ---------------------------------------------------------
# Header
# ---------------------------------------------------------

st.title("ARGUS SAT")
st.caption(
    "Supervisory Analytics & Triage Workbench"
)

st.divider()


# ---------------------------------------------------------
# Sidebar
# ---------------------------------------------------------

st.sidebar.header("Navigation")

page = st.sidebar.radio(
    "Go to",
    [
        "Dashboard",
        "Findings",
        "Data Quality"
    ]
)


# ---------------------------------------------------------
# Dashboard
# ---------------------------------------------------------

if page == "Dashboard":

    st.header("Supervisory Dashboard")

    columns, rows = get_findings()

    findings_df = pd.DataFrame(
        rows,
        columns=columns
    )

    col1, col2, col3, col4 = st.columns(4)

    col1.metric(
        "Total Findings",
        len(findings_df)
    )

    if len(findings_df) > 0:

        col2.metric(
            "High",
            len(
                findings_df[
                    findings_df["severity"] == "HIGH"
                ]
            )
        )

        col3.metric(
            "Medium",
            len(
                findings_df[
                    findings_df["severity"] == "MEDIUM"
                ]
            )
        )

        col4.metric(
            "Avg Priority",
            round(
                findings_df["priority_score"].mean(),
                1
            )
        )

    else:

        col2.metric("High", 0)
        col3.metric("Medium", 0)
        col4.metric("Avg Priority", 0)

    st.divider()

    if len(findings_df) > 0:

        st.subheader("Findings by Rule")

        rule_counts = (
            findings_df["rule_id"]
            .value_counts()
            .rename_axis("Rule")
            .reset_index(name="Findings")
        )

        st.bar_chart(
            rule_counts.set_index("Rule")
        )

    else:

        st.info(
            "No findings are currently stored."
        )


# ---------------------------------------------------------
# Findings
# ---------------------------------------------------------

elif page == "Findings":

    st.header("Supervisory Findings")

    columns, rows = get_findings()

    findings_df = pd.DataFrame(
        rows,
        columns=columns
    )

    if len(findings_df) == 0:

        st.info("No findings available.")

    else:

        st.dataframe(
            findings_df,
            use_container_width=True,
            hide_index=True
        )

        st.divider()

        st.subheader("Finding Details")

        selected_id = st.selectbox(
            "Select finding",
            findings_df["finding_id"]
        )

        selected = findings_df[
            findings_df["finding_id"] == selected_id
        ].iloc[0]

        st.write(
            "**Entity:**",
            selected["entity_id"]
        )

        st.write(
            "**Rule:**",
            selected["rule_id"]
        )

        st.write(
            "**Severity:**",
            selected["severity"]
        )

        st.write(
            "**Priority:**",
            selected["priority_score"]
        )

        st.write(
            "**Explanation:**",
            selected["explanation"]
        )

        st.write(
            "**Limitations:**",
            selected["limitations"]
        )


# ---------------------------------------------------------
# Data Quality
# ---------------------------------------------------------

elif page == "Data Quality":

    st.header("Data Quality")

    st.info(
        "Data validation results will appear here "
        "after the ingestion workflow is connected."
    )

Writing app.py


In [115]:
!pip install -q pyngrok

In [116]:
# @title
import subprocess
import time

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "app.py",
        "--server.port",
        "8501",
        "--server.address",
        "0.0.0.0"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

print("Streamlit server started.")

Streamlit server started.


In [117]:
%%writefile analytics/pipeline.py

import uuid
import pandas as pd

from ingestion.import_data import load_file
from ingestion.validate_data import validate_dataframe
from database.schema import initialize_database
from database.findings_repository import save_findings
from analytics.run_rules import run_all_rules


def run_analysis(
    cases_path,
    alerts_path,
    assets_path
):
    """
    Complete offline ARGUS SAT analysis pipeline.
    """

    initialize_database()

    # --------------------------------------------------
    # 1. Load evidence
    # --------------------------------------------------

    cases = load_file(cases_path)
    alerts = load_file(alerts_path)
    assets = load_file(assets_path)

    # --------------------------------------------------
    # 2. Validate evidence
    # --------------------------------------------------

    cases_validation = validate_dataframe(
        cases,
        required_columns=[
            "case_id",
            "entity_id",
            "severity",
            "status",
            "opened_at",
            "closed_at"
        ],
        id_column="case_id",
        timestamp_columns=[
            "opened_at",
            "closed_at"
        ]
    )

    alerts_validation = validate_dataframe(
        alerts,
        required_columns=[
            "alert_id",
            "entity_id",
            "severity",
            "opened_at",
            "status"
        ],
        id_column="alert_id",
        timestamp_columns=[
            "opened_at"
        ]
    )

    assets_validation = validate_dataframe(
        assets,
        required_columns=[
            "asset_id",
            "entity_id",
            "asset_criticality",
            "monitoring_expected",
            "asset_type"
        ],
        id_column="asset_id"
    )

    validations = {
        "cases": cases_validation,
        "alerts": alerts_validation,
        "assets": assets_validation
    }

    # --------------------------------------------------
    # 3. Stop if critical validation errors exist
    # --------------------------------------------------

    for name, result in validations.items():

        if not result["valid"]:

            raise ValueError(
                f"Validation failed for {name}: "
                f"{result['errors'][:5]}"
            )

    # --------------------------------------------------
    # 4. Run supervisory rules
    # --------------------------------------------------

    findings = run_all_rules(
        cases=cases,
        alerts=alerts,
        assets=assets,
        escalations=[],
        investigation_actions=[],
        telemetry=[]
    )

    # --------------------------------------------------
    # 5. Generate unique analytics run ID
    # --------------------------------------------------

    analytics_run_id = (
        "RUN-" + uuid.uuid4().hex[:12].upper()
    )

    # --------------------------------------------------
    # 6. Persist findings
    # --------------------------------------------------

    save_findings(
        findings,
        analytics_run_id
    )

    return {
        "analytics_run_id": analytics_run_id,
        "validations": validations,
        "findings": findings
    }

Writing analytics/pipeline.py


In [119]:
from analytics.pipeline import run_analysis

result = run_analysis(
    cases_path="synthetic_data/output/cases.csv",
    alerts_path="synthetic_data/output/alerts.csv",
    assets_path="synthetic_data/output/assets.csv"
)

print(
    "Analytics Run:",
    result["analytics_run_id"]
)

print(
    "Total Findings:",
    len(result["findings"])
)

ARGUS SAT database initialized.
Analytics Run: RUN-4C00233260B2
Total Findings: 170


In [ ]:
%%writefile review/reviewer.py

from datetime import datetime, timezone
import uuid

from database.schema import get_connection


VALID_OUTCOMES = {
    "Corroborated",
    "Not Corroborated",
    "Insufficient Evidence"
}


def record_review(
    finding_id,
    reviewer,
    outcome,
    notes=""
):

    if outcome not in VALID_OUTCOMES:
        raise ValueError(
            f"Invalid outcome: {outcome}"
        )

    conn = get_connection()

    current = conn.execute(
        """
        SELECT status
        FROM findings
        WHERE finding_id = ?
        """,
        (finding_id,)
    ).fetchone()

    if current is None:
        conn.close()
        raise ValueError(
            f"Finding not found: {finding_id}"
        )

    previous_status = current[0]

    conn.execute(
        """
        UPDATE findings
        SET status = ?
        WHERE finding_id = ?
        """,
        (
            outcome,
            finding_id
        )
    )

    event_id = (
        "AUDIT-" +
        uuid.uuid4().hex[:12].upper()
    )

    conn.execute(
        """
        INSERT INTO audit_events (
            event_id,
            actor,
            event_type,
            target_type,
            target_id,
            timestamp,
            previous_status,
            new_status,
            notes
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            event_id,
            reviewer,
            "REVIEW_DECISION",
            "FINDING",
            finding_id,
            datetime.now(
                timezone.utc
            ).isoformat(),
            previous_status,
            outcome,
            notes
        )
    )

    conn.commit()
    conn.close()

    return event_id

In [120]:
from database.findings_repository import get_findings

columns, rows = get_findings()

findings_df = pd.DataFrame(
    rows,
    columns=columns
)

findings_df[
    [
        "finding_id",
        "rule_id",
        "severity",
        "status"
    ]
].head()

,finding_id,rule_id,severity,status
0,R5-AST-0001,R5,HIGH,New
1,R5-AST-0002,R5,HIGH,New
2,R5-AST-0003,R5,HIGH,New
3,R5-AST-0004,R5,HIGH,New
4,R5-AST-0005,R5,HIGH,New


In [121]:
finding_id = findings_df.iloc[0]["finding_id"]

print("Selected:", finding_id)

Selected: R5-AST-0001


In [122]:
from review.reviewer import record_review

event_id = record_review(
    finding_id=finding_id,
    reviewer="Supervisor-001",
    outcome="Insufficient Evidence",
    notes=(
        "Initial automated evidence was insufficient "
        "for a supervisory conclusion. Additional "
        "records should be requested."
    )
)

print("Audit event created:", event_id)

Audit event created: AUDIT-C7EDC763A0C8


In [123]:
columns, rows = get_findings()

updated_df = pd.DataFrame(
    rows,
    columns=columns
)

display(
    updated_df[
        updated_df["finding_id"] == finding_id
    ][
        [
            "finding_id",
            "rule_id",
            "status"
        ]
    ]
)

,finding_id,rule_id,status
0,R5-AST-0001,R5,Insufficient Evidence


In [124]:
%%writefile app.py

import streamlit as st
import pandas as pd

from database.schema import initialize_database
from database.findings_repository import get_findings
from review.reviewer import record_review


# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

st.set_page_config(
    page_title="ARGUS SAT",
    page_icon="🔎",
    layout="wide"
)

initialize_database()


# ---------------------------------------------------------
# Header
# ---------------------------------------------------------

st.title("ARGUS SAT")
st.caption(
    "Supervisory Analytics & Triage Workbench"
)

st.divider()


# ---------------------------------------------------------
# Sidebar
# ---------------------------------------------------------

st.sidebar.header("ARGUS SAT")

page = st.sidebar.radio(
    "Navigation",
    [
        "Dashboard",
        "Findings",
        "Review Queue",
        "Data Quality"
    ]
)


# ---------------------------------------------------------
# Load findings
# ---------------------------------------------------------

columns, rows = get_findings()

findings_df = pd.DataFrame(
    rows,
    columns=columns
)


# =========================================================
# DASHBOARD
# =========================================================

if page == "Dashboard":

    st.header("Supervisory Dashboard")

    col1, col2, col3, col4 = st.columns(4)

    col1.metric(
        "Total Findings",
        len(findings_df)
    )

    if len(findings_df) > 0:

        col2.metric(
            "High",
            len(
                findings_df[
                    findings_df["severity"] == "HIGH"
                ]
            )
        )

        col3.metric(
            "Medium",
            len(
                findings_df[
                    findings_df["severity"] == "MEDIUM"
                ]
            )
        )

        col4.metric(
            "Open",
            len(
                findings_df[
                    findings_df["status"] == "New"
                ]
            )
        )

    else:

        col2.metric("High", 0)
        col3.metric("Medium", 0)
        col4.metric("Open", 0)

    st.divider()

    if len(findings_df) > 0:

        st.subheader("Findings by Rule")

        rule_counts = (
            findings_df["rule_id"]
            .value_counts()
        )

        st.bar_chart(rule_counts)

        st.subheader(
            "Priority-ordered Findings"
        )

        st.dataframe(
            findings_df[
                [
                    "finding_id",
                    "entity_id",
                    "rule_id",
                    "severity",
                    "priority_score",
                    "status"
                ]
            ],
            use_container_width=True,
            hide_index=True
        )

    else:

        st.info(
            "No findings available."
        )


# =========================================================
# FINDINGS
# =========================================================

elif page == "Findings":

    st.header("Findings Explorer")

    if len(findings_df) == 0:

        st.info("No findings available.")

    else:

        severity_filter = st.multiselect(
            "Severity",
            options=sorted(
                findings_df["severity"].unique()
            ),
            default=sorted(
                findings_df["severity"].unique()
            )
        )

        filtered = findings_df[
            findings_df["severity"].isin(
                severity_filter
            )
        ]

        st.dataframe(
            filtered,
            use_container_width=True,
            hide_index=True
        )


# =========================================================
# REVIEW QUEUE
# =========================================================

elif page == "Review Queue":

    st.header("Supervisor Review Queue")

    if len(findings_df) == 0:

        st.info("No findings available.")

    else:

        open_findings = findings_df[
            findings_df["status"] == "New"
        ]

        st.metric(
            "Awaiting Review",
            len(open_findings)
        )

        if len(open_findings) == 0:

            st.success(
                "No findings are currently awaiting review."
            )

        else:

            selected_id = st.selectbox(
                "Select finding",
                open_findings["finding_id"]
            )

            selected = open_findings[
                open_findings["finding_id"]
                == selected_id
            ].iloc[0]

            st.subheader(
                selected["finding_id"]
            )

            st.write(
                "**Entity:**",
                selected["entity_id"]
            )

            st.write(
                "**Rule:**",
                selected["rule_id"]
            )

            st.write(
                "**Severity:**",
                selected["severity"]
            )

            st.write(
                "**Priority:**",
                selected["priority_score"]
            )

            st.write(
                "**Explanation:**",
                selected["explanation"]
            )

            st.warning(
                selected["limitations"]
            )

            st.divider()

            reviewer = st.text_input(
                "Reviewer",
                value="Supervisor-001"
            )

            outcome = st.selectbox(
                "Review outcome",
                [
                    "Corroborated",
                    "Not Corroborated",
                    "Insufficient Evidence"
                ]
            )

            notes = st.text_area(
                "Review notes"
            )

            if st.button(
                "Record Decision"
            ):

                if not reviewer.strip():

                    st.error(
                        "Reviewer name is required."
                    )

                else:

                    event_id = record_review(
                        finding_id=selected_id,
                        reviewer=reviewer,
                        outcome=outcome,
                        notes=notes
                    )

                    st.success(
                        f"Decision recorded. "
                        f"Audit event: {event_id}"
                    )

                    st.rerun()


# =========================================================
# DATA QUALITY
# =========================================================

elif page == "Data Quality":

    st.header("Data Quality")

    st.info(
        "The next version will connect this page "
        "directly to the import validation results."
    )

Overwriting app.py


In [125]:
!python -m py_compile app.py

In [128]:
!python -m py_compile analytics/pipeline.py
!python -m py_compile analytics/rules.py
!python -m py_compile review/reviewer.py

print("All major modules compile successfully.")

[Errno 2] No such file or directory: 'review/reviewer.py'All major modules compile successfully.


In [49]:
%%writefile ingestion/import_service.py

import os
import shutil
import uuid
from pathlib import Path

import pandas as pd

from ingestion.import_data import load_file
from ingestion.validate_data import validate_dataframe
from evidence.provenance import create_import_metadata
from database.schema import get_connection


UPLOAD_DIR = Path("data/uploads")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)


def save_uploaded_file(uploaded_file):

    file_id = uuid.uuid4().hex[:12]

    safe_name = Path(uploaded_file.name).name

    destination = (
        UPLOAD_DIR /
        f"{file_id}_{safe_name}"
    )

    with open(destination, "wb") as file:
        file.write(uploaded_file.getbuffer())

    return destination


def validate_uploaded_file(
    file_path,
    dataset_type
):

    df = load_file(file_path)

    schemas = {

        "Cases": {
            "required": [
                "case_id",
                "entity_id",
                "severity",
                "status",
                "opened_at",
                "closed_at"
            ],
            "id": "case_id",
            "timestamps": [
                "opened_at",
                "closed_at"
            ]
        },

        "Alerts": {
            "required": [
                "alert_id",
                "entity_id",
                "severity",
                "opened_at",
                "status"
            ],
            "id": "alert_id",
            "timestamps": [
                "opened_at"
            ]
        },

        "Assets": {
            "required": [
                "asset_id",
                "entity_id",
                "asset_criticality",
                "monitoring_expected",
                "asset_type"
            ],
            "id": "asset_id",
            "timestamps": []
        },

        "Escalations": {
            "required": [
                "escalation_id",
                "entity_id",
                "case_id",
                "escalated_at",
                "escalation_type",
                "status"
            ],
            "id": "escalation_id",
            "timestamps": [
                "escalated_at"
            ]
        },

        "Investigation Actions": {
            "required": [
                "action_id",
                "entity_id",
                "case_id",
                "action_type",
                "action_at",
                "status"
            ],
            "id": "action_id",
            "timestamps": [
                "action_at"
            ]
        },

        "Telemetry": {
            "required": [
                "telemetry_id",
                "entity_id",
                "asset_id",
                "observed_at",
                "metric",
                "value"
            ],
            "id": "telemetry_id",
            "timestamps": [
                "observed_at"
            ]
        }
    }

    if dataset_type not in schemas:
        raise ValueError(
            f"Unknown dataset type: {dataset_type}"
        )

    schema = schemas[dataset_type]

    validation = validate_dataframe(
        df,
        required_columns=schema["required"],
        id_column=schema["id"],
        timestamp_columns=schema["timestamps"]
    )

    return df, validation


def save_import_batch(
    file_path,
    entity_id,
    validation
):

    metadata = create_import_metadata(
        file_path=file_path,
        entity_id=entity_id,
        reporting_period_start="2026-01-01",
        reporting_period_end="2026-03-31"
    )

    conn = get_connection()

    conn.execute(
        """
        INSERT OR REPLACE INTO import_batches (
            submission_id,
            entity_id,
            reporting_period_start,
            reporting_period_end,
            source_filename,
            imported_at,
            file_hash,
            accepted_records,
            rejected_records
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            metadata["submission_id"],
            metadata["entity_id"],
            metadata["reporting_period_start"],
            metadata["reporting_period_end"],
            metadata["source_filename"],
            metadata["imported_at"],
            metadata["file_hash"],
            validation["row_count"]
            - len(validation["errors"]),
            len(validation["errors"])
        )
    )

    conn.commit()
    conn.close()

    return metadata

Writing ingestion/import_service.py


In [50]:
from ingestion.import_service import (
    validate_uploaded_file
)

df, validation = validate_uploaded_file(
    "synthetic_data/output/cases.csv",
    "Cases"
)

print("Rows:", validation["row_count"])
print("Valid:", validation["valid"])
print("Errors:", len(validation["errors"]))
print("Warnings:", len(validation["warnings"]))

Rows: 100
Valid: True
Errors: 0
Warnings: 0


In [51]:
%%writefile app.py

import streamlit as st
import pandas as pd

from database.schema import initialize_database
from database.findings_repository import get_findings
from review.reviewer import record_review

from ingestion.import_service import (
    save_uploaded_file,
    validate_uploaded_file,
    save_import_batch
)

from analytics.pipeline import run_analysis


# =========================================================
# CONFIG
# =========================================================

st.set_page_config(
    page_title="ARGUS SAT",
    page_icon="🔎",
    layout="wide"
)

initialize_database()


# =========================================================
# HEADER
# =========================================================

st.title("ARGUS SAT")

st.caption(
    "Supervisory Analytics & Triage Workbench"
)

st.divider()


# =========================================================
# NAVIGATION
# =========================================================

page = st.sidebar.radio(
    "Navigation",
    [
        "Dashboard",
        "Import & Validate",
        "Findings",
        "Review Queue",
        "Data Quality"
    ]
)


# =========================================================
# DASHBOARD
# =========================================================

if page == "Dashboard":

    st.header("Supervisory Dashboard")

    columns, rows = get_findings()

    findings_df = pd.DataFrame(
        rows,
        columns=columns
    )

    c1, c2, c3, c4 = st.columns(4)

    c1.metric(
        "Total Findings",
        len(findings_df)
    )

    c2.metric(
        "High",
        len(
            findings_df[
                findings_df["severity"] == "HIGH"
            ]
        )
    )

    c3.metric(
        "Medium",
        len(
            findings_df[
                findings_df["severity"] == "MEDIUM"
            ]
        )
    )

    c4.metric(
        "Awaiting Review",
        len(
            findings_df[
                findings_df["status"] == "New"
            ]
        )
    )

    st.divider()

    if len(findings_df) > 0:

        st.subheader(
            "Findings by Rule"
        )

        st.bar_chart(
            findings_df["rule_id"]
            .value_counts()
        )

        st.subheader(
            "Priority Queue"
        )

        st.dataframe(
            findings_df[
                [
                    "finding_id",
                    "entity_id",
                    "rule_id",
                    "severity",
                    "priority_score",
                    "status"
                ]
            ],
            use_container_width=True,
            hide_index=True
        )

    else:

        st.info(
            "No findings are currently available."
        )


# =========================================================
# IMPORT & VALIDATE
# =========================================================

elif page == "Import & Validate":

    st.header(
        "Evidence Import & Validation"
    )

    st.write(
        "Upload a CSV or JSON evidence file, "
        "validate it, inspect errors, and register "
        "its provenance."
    )

    dataset_type = st.selectbox(
        "Dataset type",
        [
            "Cases",
            "Alerts",
            "Assets"
        ]
    )

    uploaded_file = st.file_uploader(
        "Evidence file",
        type=["csv", "json"]
    )

    entity_id = st.text_input(
        "Entity ID",
        value="ORG-001"
    )

    if uploaded_file:

        if st.button(
            "Validate Evidence"
        ):

            try:

                path = save_uploaded_file(
                    uploaded_file
                )

                df, validation = (
                    validate_uploaded_file(
                        path,
                        dataset_type
                    )
                )

                st.session_state[
                    "last_uploaded_path"
                ] = str(path)

                st.session_state[
                    "last_validation"
                ] = validation

                st.session_state[
                    "last_dataset_type"
                ] = dataset_type

                st.session_state[
                    "last_entity_id"
                ] = entity_id

                if validation["valid"]:

                    st.success(
                        "Validation passed."
                    )

                else:

                    st.error(
                        "Validation failed."
                    )

                c1, c2, c3 = st.columns(3)

                c1.metric(
                    "Rows",
                    validation["row_count"]
                )

                c2.metric(
                    "Errors",
                    len(validation["errors"])
                )

                c3.metric(
                    "Warnings",
                    len(validation["warnings"])
                )

                st.subheader(
                    "Preview"
                )

                st.dataframe(
                    df.head(100),
                    use_container_width=True,
                    hide_index=True
                )

                if validation["errors"]:

                    st.subheader(
                        "Validation Errors"
                    )

                    for error in validation[
                        "errors"
                    ]:

                        st.error(error)

                if validation["warnings"]:

                    st.subheader(
                        "Warnings"
                    )

                    for warning in validation[
                        "warnings"
                    ]:

                        st.warning(warning)

                if validation["valid"]:

                    metadata = (
                        save_import_batch(
                            path,
                            entity_id,
                            validation
                        )
                    )

                    st.success(
                        "Import batch registered."
                    )

                    st.json(metadata)

            except Exception as error:

                st.error(
                    f"Import failed: {error}"
                )


# =========================================================
# FINDINGS
# =========================================================

elif page == "Findings":

    st.header(
        "Findings Explorer"
    )

    columns, rows = get_findings()

    findings_df = pd.DataFrame(
        rows,
        columns=columns
    )

    if len(findings_df) == 0:

        st.info(
            "No findings available."
        )

    else:

        severity = st.multiselect(
            "Severity",
            sorted(
                findings_df["severity"]
                .unique()
            ),
            default=sorted(
                findings_df["severity"]
                .unique()
            )
        )

        filtered = findings_df[
            findings_df["severity"].isin(
                severity
            )
        ]

        st.dataframe(
            filtered,
            use_container_width=True,
            hide_index=True
        )


# =========================================================
# REVIEW QUEUE
# =========================================================

elif page == "Review Queue":

    st.header(
        "Supervisor Review Queue"
    )

    columns, rows = get_findings()

    findings_df = pd.DataFrame(
        rows,
        columns=columns
    )

    open_findings = findings_df[
        findings_df["status"] == "New"
    ]

    st.metric(
        "Awaiting Review",
        len(open_findings)
    )

    if len(open_findings) == 0:

        st.success(
            "No findings awaiting review."
        )

    else:

        selected_id = st.selectbox(
            "Finding",
            open_findings["finding_id"]
        )

        selected = open_findings[
            open_findings["finding_id"]
            == selected_id
        ].iloc[0]

        st.subheader(
            selected["finding_id"]
        )

        st.write(
            "**Entity:**",
            selected["entity_id"]
        )

        st.write(
            "**Rule:**",
            selected["rule_id"]
        )

        st.write(
            "**Severity:**",
            selected["severity"]
        )

        st.write(
            "**Priority:**",
            selected["priority_score"]
        )

        st.write(
            "**Explanation:**",
            selected["explanation"]
        )

        st.warning(
            selected["limitations"]
        )

        reviewer = st.text_input(
            "Reviewer",
            "Supervisor-001"
        )

        outcome = st.selectbox(
            "Decision",
            [
                "Corroborated",
                "Not Corroborated",
                "Insufficient Evidence"
            ]
        )

        notes = st.text_area(
            "Review notes"
        )

        if st.button(
            "Record Decision"
        ):

            event_id = record_review(
                finding_id=selected_id,
                reviewer=reviewer,
                outcome=outcome,
                notes=notes
            )

            st.success(
                f"Decision recorded: {event_id}"
            )

            st.rerun()


# =========================================================
# DATA QUALITY
# =========================================================

elif page == "Data Quality":

    st.header(
        "Data Quality Overview"
    )

    st.info(
        "Use Import & Validate to inspect "
        "incoming evidence."
    )

Overwriting app.py


In [52]:
!python -m py_compile app.py

In [53]:
!python -m py_compile \
    ingestion/import_service.py \
    ingestion/import_data.py \
    ingestion/validate_data.py \
    analytics/rules.py \
    analytics/run_rules.py \
    analytics/pipeline.py \
    analytics/negative_space.py \
    analytics/prioritization.py \
    database/schema.py \
    database/repository.py \
    database/findings_repository.py \
    evidence/provenance.py \
    review/reviewer.py

print("ARGUS SAT modules compiled successfully.")

ARGUS SAT modules compiled successfully.


In [54]:
from ingestion.import_service import (
    validate_uploaded_file
)

df, result = validate_uploaded_file(
    "synthetic_data/output/cases.csv",
    "Cases"
)

print("VALID:", result["valid"])
print("ROWS:", result["row_count"])
print("ERRORS:", result["errors"])

VALID: True
ROWS: 100
ERRORS: []


In [55]:
%%writefile app.py

import sqlite3
import streamlit as st
import pandas as pd

from database.schema import initialize_database
from database.findings_repository import get_findings
from review.reviewer import record_review


# =========================================================
# CONFIG
# =========================================================

st.set_page_config(
    page_title="ARGUS SAT",
    page_icon="🔎",
    layout="wide",
    initial_sidebar_state="expanded"
)

initialize_database()


# =========================================================
# DATABASE HELPERS
# =========================================================

def get_audit_events():

    conn = sqlite3.connect("argus_sat.db")

    rows = conn.execute(
        """
        SELECT
            event_id,
            actor,
            event_type,
            target_type,
            target_id,
            timestamp,
            previous_status,
            new_status,
            notes
        FROM audit_events
        ORDER BY timestamp DESC
        """
    ).fetchall()

    columns = [
        "event_id",
        "actor",
        "event_type",
        "target_type",
        "target_id",
        "timestamp",
        "previous_status",
        "new_status",
        "notes"
    ]

    conn.close()

    return pd.DataFrame(
        rows,
        columns=columns
    )


def get_finding_evidence(finding_id):

    conn = sqlite3.connect("argus_sat.db")

    rows = conn.execute(
        """
        SELECT
            source_record_type,
            source_record_id,
            source_file_id,
            source_row_or_json_pointer,
            evidence_role
        FROM finding_evidence
        WHERE finding_id = ?
        """,
        (finding_id,)
    ).fetchall()

    columns = [
        "source_record_type",
        "source_record_id",
        "source_file_id",
        "source_row_or_json_pointer",
        "evidence_role"
    ]

    conn.close()

    return pd.DataFrame(
        rows,
        columns=columns
    )


# =========================================================
# LOAD FINDINGS
# =========================================================

columns, rows = get_findings()

findings_df = pd.DataFrame(
    rows,
    columns=columns
)


# =========================================================
# SIDEBAR
# =========================================================

st.sidebar.title("ARGUS SAT")

st.sidebar.caption(
    "Supervisory Analytics & Triage"
)

st.sidebar.divider()

page = st.sidebar.radio(
    "Workspace",
    [
        "Overview",
        "Findings Explorer",
        "Review Queue",
        "Audit Trail",
        "Data Quality"
    ]
)

st.sidebar.divider()

st.sidebar.caption(
    "Offline analytical prototype"
)


# =========================================================
# OVERVIEW
# =========================================================

if page == "Overview":

    st.title("Supervisory Overview")

    st.caption(
        "Evidence-driven analytics for supervisory triage"
    )

    # -----------------------------------------------------
    # KPI ROW
    # -----------------------------------------------------

    total = len(findings_df)

    high = len(
        findings_df[
            findings_df["severity"] == "HIGH"
        ]
    )

    medium = len(
        findings_df[
            findings_df["severity"] == "MEDIUM"
        ]
    )

    open_count = len(
        findings_df[
            findings_df["status"] == "New"
        ]
    )

    c1, c2, c3, c4 = st.columns(4)

    c1.metric(
        "Total Findings",
        total
    )

    c2.metric(
        "High Severity",
        high
    )

    c3.metric(
        "Medium Severity",
        medium
    )

    c4.metric(
        "Awaiting Review",
        open_count
    )

    st.divider()

    if total == 0:

        st.info(
            "No findings have been generated yet."
        )

    else:

        # -------------------------------------------------
        # CHARTS
        # -------------------------------------------------

        left, right = st.columns(2)

        with left:

            st.subheader(
                "Severity Distribution"
            )

            severity_counts = (
                findings_df[
                    "severity"
                ]
                .value_counts()
            )

            st.bar_chart(
                severity_counts
            )

        with right:

            st.subheader(
                "Rule Distribution"
            )

            rule_counts = (
                findings_df[
                    "rule_id"
                ]
                .value_counts()
            )

            st.bar_chart(
                rule_counts
            )

        st.divider()

        # -------------------------------------------------
        # ENTITY VIEW
        # -------------------------------------------------

        st.subheader(
            "Findings by Entity"
        )

        entity_counts = (
            findings_df[
                "entity_id"
            ]
            .value_counts()
            .rename("findings")
            .reset_index()
        )

        entity_counts.columns = [
            "entity_id",
            "findings"
        ]

        st.dataframe(
            entity_counts,
            use_container_width=True,
            hide_index=True
        )

        st.divider()

        # -------------------------------------------------
        # PRIORITY QUEUE
        # -------------------------------------------------

        st.subheader(
            "Priority Queue"
        )

        priority_view = findings_df[
            [
                "finding_id",
                "entity_id",
                "rule_id",
                "severity",
                "priority_score",
                "evidence_strength",
                "status"
            ]
        ].sort_values(
            "priority_score",
            ascending=False
        )

        st.dataframe(
            priority_view,
            use_container_width=True,
            hide_index=True
        )


# =========================================================
# FINDINGS EXPLORER
# =========================================================

elif page == "Findings Explorer":

    st.title(
        "Findings Explorer"
    )

    if findings_df.empty:

        st.info(
            "No findings available."
        )

    else:

        # -------------------------------------------------
        # FILTERS
        # -------------------------------------------------

        c1, c2, c3 = st.columns(3)

        with c1:

            severity_filter = st.multiselect(
                "Severity",
                sorted(
                    findings_df[
                        "severity"
                    ].unique()
                ),
                default=sorted(
                    findings_df[
                        "severity"
                    ].unique()
                )
            )

        with c2:

            rule_filter = st.multiselect(
                "Rule",
                sorted(
                    findings_df[
                        "rule_id"
                    ].unique()
                ),
                default=sorted(
                    findings_df[
                        "rule_id"
                    ].unique()
                )
            )

        with c3:

            status_filter = st.multiselect(
                "Status",
                sorted(
                    findings_df[
                        "status"
                    ].unique()
                ),
                default=sorted(
                    findings_df[
                        "status"
                    ].unique()
                )
            )

        filtered = findings_df[
            findings_df["severity"].isin(
                severity_filter
            )
            &
            findings_df["rule_id"].isin(
                rule_filter
            )
            &
            findings_df["status"].isin(
                status_filter
            )
        ]

        st.write(
            f"Showing {len(filtered)} findings"
        )

        st.dataframe(
            filtered[
                [
                    "finding_id",
                    "entity_id",
                    "category",
                    "rule_id",
                    "severity",
                    "priority_score",
                    "evidence_strength",
                    "status"
                ]
            ],
            use_container_width=True,
            hide_index=True
        )

        st.divider()

        # -------------------------------------------------
        # DRILL DOWN
        # -------------------------------------------------

        selected_id = st.selectbox(
            "Open finding",
            filtered["finding_id"]
        )

        selected = filtered[
            filtered["finding_id"]
            == selected_id
        ].iloc[0]

        st.header(
            selected["finding_id"]
        )

        c1, c2, c3, c4 = st.columns(4)

        c1.metric(
            "Entity",
            selected["entity_id"]
        )

        c2.metric(
            "Rule",
            selected["rule_id"]
        )

        c3.metric(
            "Severity",
            selected["severity"]
        )

        c4.metric(
            "Priority",
            selected["priority_score"]
        )

        st.subheader(
            "Analytical Explanation"
        )

        st.write(
            selected["explanation"]
        )

        st.subheader(
            "Evidence Strength"
        )

        st.write(
            selected["evidence_strength"]
        )

        st.subheader(
            "Limitations"
        )

        st.warning(
            selected["limitations"]
        )

        # -------------------------------------------------
        # EVIDENCE
        # -------------------------------------------------

        st.subheader(
            "Evidence References"
        )

        evidence_df = get_finding_evidence(
            selected_id
        )

        if evidence_df.empty:

            st.info(
                "No evidence references stored."
            )

        else:

            st.dataframe(
                evidence_df,
                use_container_width=True,
                hide_index=True
            )


# =========================================================
# REVIEW QUEUE
# =========================================================

elif page == "Review Queue":

    st.title(
        "Supervisor Review Queue"
    )

    if findings_df.empty:

        st.info(
            "No findings available."
        )

    else:

        open_findings = findings_df[
            findings_df["status"] == "New"
        ]

        c1, c2 = st.columns(2)

        c1.metric(
            "Awaiting Review",
            len(open_findings)
        )

        c2.metric(
            "Total Findings",
            len(findings_df)
        )

        st.divider()

        if open_findings.empty:

            st.success(
                "All current findings have been reviewed."
            )

        else:

            selected_id = st.selectbox(
                "Select finding",
                open_findings["finding_id"]
            )

            selected = open_findings[
                open_findings["finding_id"]
                == selected_id
            ].iloc[0]

            st.header(
                selected["finding_id"]
            )

            st.write(
                "**Entity:**",
                selected["entity_id"]
            )

            st.write(
                "**Rule:**",
                selected["rule_id"]
            )

            st.write(
                "**Severity:**",
                selected["severity"]
            )

            st.write(
                "**Priority:**",
                selected["priority_score"]
            )

            st.subheader(
                "Finding"
            )

            st.write(
                selected["explanation"]
            )

            st.subheader(
                "Limitations"
            )

            st.warning(
                selected["limitations"]
            )

            st.subheader(
                "Evidence"
            )

            evidence_df = get_finding_evidence(
                selected_id
            )

            st.dataframe(
                evidence_df,
                use_container_width=True,
                hide_index=True
            )

            st.divider()

            reviewer = st.text_input(
                "Reviewer",
                "Supervisor-001"
            )

            outcome = st.selectbox(
                "Review outcome",
                [
                    "Corroborated",
                    "Not Corroborated",
                    "Insufficient Evidence"
                ]
            )

            notes = st.text_area(
                "Review notes"
            )

            if st.button(
                "Record Review Decision",
                type="primary"
            ):

                if not reviewer.strip():

                    st.error(
                        "Reviewer is required."
                    )

                else:

                    event_id = record_review(
                        finding_id=selected_id,
                        reviewer=reviewer,
                        outcome=outcome,
                        notes=notes
                    )

                    st.success(
                        f"Review recorded. Audit event: {event_id}"
                    )

                    st.rerun()


# =========================================================
# AUDIT TRAIL
# =========================================================

elif page == "Audit Trail":

    st.title(
        "Audit Trail"
    )

    audit_df = get_audit_events()

    if audit_df.empty:

        st.info(
            "No audit events recorded yet."
        )

    else:

        st.dataframe(
            audit_df,
            use_container_width=True,
            hide_index=True
        )


# =========================================================
# DATA QUALITY
# =========================================================

elif page == "Data Quality":

    st.title(
        "Data Quality"
    )

    st.write(
        "Use the Import & Validate workflow "
        "to inspect incoming evidence."
    )

    st.info(
        "Detailed validation metrics will be "
        "connected here next."
    )

Overwriting app.py


In [56]:
!python -m py_compile app.py
print("Dashboard compiled successfully.")

Dashboard compiled successfully.


In [57]:
!python -m py_compile \
    analytics/*.py \
    database/*.py \
    evidence/*.py \
    ingestion/*.py \
    review/*.py

print("ARGUS SAT backend compiled successfully.")

ARGUS SAT backend compiled successfully.


In [58]:
%%writefile requirements.txt

streamlit
pandas
numpy

Writing requirements.txt


In [59]:
%%writefile run_argus.py

import subprocess
import sys


def main():

    print("=" * 55)
    print("ARGUS SAT")
    print("Supervisory Analytics & Triage Workbench")
    print("=" * 55)

    print("\nStarting local application...")
    print("Open your browser at:")
    print("http://localhost:8501")
    print("\nPress Ctrl+C to stop the application.\n")

    subprocess.run(
        [
            sys.executable,
            "-m",
            "streamlit",
            "run",
            "app.py",
            "--server.address",
            "localhost",
            "--server.port",
            "8501",
            "--server.headless",
            "true"
        ]
    )


if __name__ == "__main__":
    main()

Writing run_argus.py


In [60]:
%%writefile config.py

from pathlib import Path


BASE_DIR = Path(__file__).resolve().parent

DATABASE_PATH = (
    BASE_DIR / "argus_sat.db"
)

UPLOAD_DIR = (
    BASE_DIR / "data" / "uploads"
)

OUTPUT_DIR = (
    BASE_DIR / "data" / "output"
)

SYNTHETIC_DIR = (
    BASE_DIR / "synthetic_data" / "output"
)


UPLOAD_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

Writing config.py


In [61]:
import sys
import sqlite3
import pandas
import numpy

print("Python:", sys.version)
print("SQLite:", sqlite3.sqlite_version)
print("Pandas:", pandas.__version__)
print("NumPy:", numpy.__version__)

Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
SQLite: 3.45.1
Pandas: 2.2.3
NumPy: 2.1.3


In [62]:
%%writefile self_test.py

import sqlite3
import importlib


MODULES = [
    "database.schema",
    "database.repository",
    "database.findings_repository",
    "evidence.provenance",
    "ingestion.import_data",
    "ingestion.validate_data",
    "ingestion.import_service",
    "analytics.rules",
    "analytics.run_rules",
    "analytics.negative_space",
    "analytics.prioritization",
    "analytics.pipeline",
    "review.reviewer"
]


def test_imports():

    print("\n[1] Testing Python modules...")

    failed = []

    for module in MODULES:

        try:
            importlib.import_module(module)
            print("  OK:", module)

        except Exception as error:

            print("  FAIL:", module)
            print("       ", error)

            failed.append(module)

    return failed


def test_database():

    print("\n[2] Testing SQLite...")

    conn = sqlite3.connect(
        "argus_sat.db"
    )

    tables = conn.execute(
        """
        SELECT name
        FROM sqlite_master
        WHERE type='table'
        """
    ).fetchall()

    conn.close()

    required = {
        "organizations",
        "assets",
        "alerts",
        "cases",
        "findings",
        "finding_evidence",
        "audit_events",
        "import_batches"
    }

    existing = {
        table[0]
        for table in tables
    }

    missing = required - existing

    if missing:

        print(
            "  FAIL - missing tables:",
            missing
        )

        return False

    print("  OK: SQLite schema")

    return True


def main():

    print("=" * 55)
    print("ARGUS SAT OFFLINE SELF-TEST")
    print("=" * 55)

    failed_modules = test_imports()

    database_ok = test_database()

    print("\n" + "=" * 55)

    if not failed_modules and database_ok:

        print("ALL TESTS PASSED")

    else:

        print("TESTS FAILED")

        if failed_modules:
            print(
                "Failed modules:",
                failed_modules
            )

        if not database_ok:
            print(
                "Database schema is incomplete."
            )


if __name__ == "__main__":
    main()

Writing self_test.py


In [63]:
!python self_test.py

ARGUS SAT OFFLINE SELF-TEST

[1] Testing Python modules...
  OK: database.schema
  OK: database.repository
  OK: database.findings_repository
  OK: evidence.provenance
  OK: ingestion.import_data
  OK: ingestion.validate_data
  OK: ingestion.import_service
  OK: analytics.rules
  OK: analytics.run_rules
  OK: analytics.negative_space
  OK: analytics.prioritization
  OK: analytics.pipeline
  OK: review.reviewer

[2] Testing SQLite...
  OK: SQLite schema

ALL TESTS PASSED


In [ ]:
%%writefile start_argus.bat

@echo off

echo ============================================
echo ARGUS SAT
echo Supervisory Analytics ^& Triage Workbench
echo ============================================

echo.
echo Starting ARGUS SAT...
echo.

python run_argus.py

pause

Writing start_argus.bat


In [65]:
from pathlib import Path

for folder in [
    "analytics",
    "database",
    "evidence",
    "ingestion",
    "review"
]:

    Path(folder).mkdir(
        parents=True,
        exist_ok=True
    )

    Path(
        folder,
        "__init__.py"
    ).touch()

print("Package structure ready.")

Package structure ready.


In [66]:
sqlite3.connect("argus_sat.db")

In [67]:
%%writefile database/schema.py

import sqlite3
from pathlib import Path


BASE_DIR = Path(__file__).resolve().parent.parent

DB_PATH = (
    BASE_DIR / "argus_sat.db"
)


def get_connection():

    return sqlite3.connect(
        DB_PATH
    )


def initialize_database():

    conn = get_connection()

    cursor = conn.cursor()

    cursor.executescript("""

    CREATE TABLE IF NOT EXISTS organizations (
        entity_id TEXT PRIMARY KEY,
        entity_name TEXT NOT NULL,
        sector TEXT,
        peer_group TEXT
    );

    CREATE TABLE IF NOT EXISTS assets (
        asset_id TEXT PRIMARY KEY,
        entity_id TEXT NOT NULL,
        asset_criticality TEXT,
        monitoring_expected INTEGER,
        asset_type TEXT,
        source_record_id TEXT,
        submission_id TEXT
    );

    CREATE TABLE IF NOT EXISTS alerts (
        alert_id TEXT PRIMARY KEY,
        entity_id TEXT NOT NULL,
        asset_id TEXT,
        severity TEXT,
        opened_at TEXT,
        closed_at TEXT,
        status TEXT,
        source_record_id TEXT,
        submission_id TEXT
    );

    CREATE TABLE IF NOT EXISTS cases (
        case_id TEXT PRIMARY KEY,
        entity_id TEXT NOT NULL,
        severity TEXT,
        status TEXT,
        opened_at TEXT,
        closed_at TEXT,
        source_record_id TEXT,
        submission_id TEXT
    );

    CREATE TABLE IF NOT EXISTS findings (
        finding_id TEXT PRIMARY KEY,
        entity_id TEXT,
        period_start TEXT,
        period_end TEXT,
        category TEXT,
        rule_id TEXT,
        rule_version TEXT,
        severity TEXT,
        priority_score REAL,
        evidence_strength TEXT,
        explanation TEXT,
        limitations TEXT,
        status TEXT,
        created_at TEXT,
        analytics_run_id TEXT
    );

    CREATE TABLE IF NOT EXISTS finding_evidence (
        finding_id TEXT,
        source_record_type TEXT,
        source_record_id TEXT,
        source_file_id TEXT,
        source_row_or_json_pointer TEXT,
        evidence_role TEXT
    );

    CREATE TABLE IF NOT EXISTS audit_events (
        event_id TEXT PRIMARY KEY,
        actor TEXT,
        event_type TEXT,
        target_type TEXT,
        target_id TEXT,
        timestamp TEXT,
        previous_status TEXT,
        new_status TEXT,
        notes TEXT,
        analytics_run_id TEXT,
        submission_id TEXT
    );

    CREATE TABLE IF NOT EXISTS import_batches (
        submission_id TEXT PRIMARY KEY,
        entity_id TEXT,
        reporting_period_start TEXT,
        reporting_period_end TEXT,
        source_filename TEXT,
        imported_at TEXT,
        file_hash TEXT,
        accepted_records INTEGER,
        rejected_records INTEGER
    );

    """)

    conn.commit()
    conn.close()

    print("ARGUS SAT database initialized.")

Overwriting database/schema.py


In [68]:
from database.schema import initialize_database

initialize_database()

ARGUS SAT database initialized.


In [69]:
!python self_test.py

ARGUS SAT OFFLINE SELF-TEST

[1] Testing Python modules...
  OK: database.schema
  OK: database.repository
  OK: database.findings_repository
  OK: evidence.provenance
  OK: ingestion.import_data
  OK: ingestion.validate_data
  OK: ingestion.import_service
  OK: analytics.rules
  OK: analytics.run_rules
  OK: analytics.negative_space
  OK: analytics.prioritization
  OK: analytics.pipeline
  OK: review.reviewer

[2] Testing SQLite...
  OK: SQLite schema

ALL TESTS PASSED


In [70]:
!python -m py_compile app.py run_argus.py self_test.py

In [71]:
from analytics.pipeline import run_analysis

result = run_analysis(
    cases_path="synthetic_data/output/cases.csv",
    alerts_path="synthetic_data/output/alerts.csv",
    assets_path="synthetic_data/output/assets.csv"
)

print("RUN:", result["analytics_run_id"])
print("FINDINGS:", len(result["findings"]))

ARGUS SAT database initialized.
RUN: RUN-7764DD8183E5
FINDINGS: 170


In [72]:
%%writefile synthetic_data/demo_scenario.py

from pathlib import Path
import pandas as pd


OUTPUT_DIR = Path("synthetic_data/demo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def create_demo_scenario():

    # =====================================================
    # CASES
    # =====================================================

    cases = pd.DataFrame([
        {
            "case_id": "CASE-R1-001",
            "entity_id": "ORG-001",
            "severity": "CRITICAL",
            "status": "CLOSED",
            "opened_at": "2026-03-01 10:00:00",
            "closed_at": "2026-03-01 10:08:00"
        },
        {
            "case_id": "CASE-R2-001",
            "entity_id": "ORG-001",
            "severity": "HIGH",
            "status": "CLOSED",
            "opened_at": "2026-03-02 09:00:00",
            "closed_at": "2026-03-02 13:00:00"
        },
        {
            "case_id": "CASE-R3-001",
            "entity_id": "ORG-002",
            "severity": "HIGH",
            "status": "CLOSED",
            "opened_at": "2026-03-03 11:00:00",
            "closed_at": "2026-03-03 16:00:00"
        },
        {
            "case_id": "CASE-NORMAL-001",
            "entity_id": "ORG-002",
            "severity": "MEDIUM",
            "status": "CLOSED",
            "opened_at": "2026-03-04 10:00:00",
            "closed_at": "2026-03-04 15:00:00"
        }
    ])

    # =====================================================
    # ALERTS
    # =====================================================

    alerts = pd.DataFrame([
        {
            "alert_id": "ALERT-R4-001",
            "entity_id": "ORG-003",
            "asset_id": "ASSET-003",
            "severity": "MEDIUM",
            "opened_at": "2026-03-01 08:00:00",
            "status": "CLOSED"
        },
        {
            "alert_id": "ALERT-R4-002",
            "entity_id": "ORG-003",
            "asset_id": "ASSET-003",
            "severity": "MEDIUM",
            "opened_at": "2026-03-05 08:00:00",
            "status": "CLOSED"
        },
        {
            "alert_id": "ALERT-R4-003",
            "entity_id": "ORG-003",
            "asset_id": "ASSET-003",
            "severity": "HIGH",
            "opened_at": "2026-03-10 08:00:00",
            "status": "CLOSED"
        },
        {
            "alert_id": "ALERT-R4-004",
            "entity_id": "ORG-003",
            "asset_id": "ASSET-003",
            "severity": "HIGH",
            "opened_at": "2026-03-15 08:00:00",
            "status": "CLOSED"
        }
    ])

    # =====================================================
    # ASSETS
    # =====================================================

    assets = pd.DataFrame([
        {
            "asset_id": "ASSET-001",
            "entity_id": "ORG-001",
            "asset_criticality": "CRITICAL",
            "monitoring_expected": True,
            "asset_type": "Production System"
        },
        {
            "asset_id": "ASSET-002",
            "entity_id": "ORG-002",
            "asset_criticality": "HIGH",
            "monitoring_expected": True,
            "asset_type": "Network System"
        },
        {
            "asset_id": "ASSET-003",
            "entity_id": "ORG-003",
            "asset_criticality": "MEDIUM",
            "monitoring_expected": True,
            "asset_type": "Application"
        }
    ])

    # =====================================================
    # SAVE
    # =====================================================

    cases.to_csv(
        OUTPUT_DIR / "cases.csv",
        index=False
    )

    alerts.to_csv(
        OUTPUT_DIR / "alerts.csv",
        index=False
    )

    assets.to_csv(
        OUTPUT_DIR / "assets.csv",
        index=False
    )

    print("Demo scenario created.")
    print("Location:", OUTPUT_DIR)


if __name__ == "__main__":
    create_demo_scenario()

Writing synthetic_data/demo_scenario.py


In [73]:
from synthetic_data.demo_scenario import create_demo_scenario

create_demo_scenario()

Demo scenario created.
Location: synthetic_data/demo


In [74]:
%%writefile demo.py

import os
import sqlite3

from database.schema import initialize_database
from analytics.pipeline import run_analysis


def reset_database():

    if os.path.exists("argus_sat.db"):
        os.remove("argus_sat.db")

    initialize_database()


def main():

    print("=" * 60)
    print("ARGUS SAT DEMONSTRATION")
    print("=" * 60)

    reset_database()

    result = run_analysis(
        cases_path="synthetic_data/demo/cases.csv",
        alerts_path="synthetic_data/demo/alerts.csv",
        assets_path="synthetic_data/demo/assets.csv"
    )

    print()
    print("Analytics run:")
    print(result["analytics_run_id"])

    print()
    print("Findings generated:")
    print(len(result["findings"]))

    print()
    print("Rule summary:")

    counts = {}

    for finding in result["findings"]:

        rule = finding["rule_id"]

        counts[rule] = (
            counts.get(rule, 0) + 1
        )

    for rule, count in sorted(
        counts.items()
    ):

        print(
            f"  {rule}: {count}"
        )

    print()
    print("=" * 60)
    print("Demo data is ready.")
    print("Run: python run_argus.py")
    print("=" * 60)


if __name__ == "__main__":
    main()

Writing demo.py


In [75]:
!python demo.py

ARGUS SAT DEMONSTRATION
ARGUS SAT database initialized.
ARGUS SAT database initialized.

Analytics run:
RUN-D024E531CF4B

Findings generated:
12

Rule summary:
  R1: 1
  R2: 3
  R3: 4
  R4: 1
  R5: 3

Demo data is ready.
Run: python run_argus.py


In [ ]:
escalations=[]
investigation_actions=[]
telemetry=[]

In [77]:
st.title("ARGUS SAT")

NameError: name 'st' is not defined

In [ ]:
!python demo.py

In [ ]:
!python self_test.py

In [ ]:
!python -m py_compile app.py demo.py run_argus.py

In [ ]:
%%writefile synthetic_data/demo/evidence.py

from pathlib import Path
import pandas as pd


OUTPUT_DIR = Path("synthetic_data/demo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def create_demo_evidence():

    # -----------------------------------------------------
    # ESCALATIONS
    # -----------------------------------------------------

    escalations = pd.DataFrame([
        {
            "escalation_id": "ESC-001",
            "entity_id": "ORG-001",
            "case_id": "CASE-R2-001",
            "escalated_at": "2026-03-02 09:15:00",
            "escalation_type": "HIGH_RISK",
            "status": "OPEN"
        },
        {
            "escalation_id": "ESC-002",
            "entity_id": "ORG-002",
            "case_id": "CASE-R3-001",
            "escalated_at": "2026-03-03 11:20:00",
            "escalation_type": "HIGH_RISK",
            "status": "OPEN"
        }
    ])

    # -----------------------------------------------------
    # INVESTIGATION ACTIONS
    # -----------------------------------------------------

    investigation_actions = pd.DataFrame([
        {
            "action_id": "INV-001",
            "entity_id": "ORG-002",
            "case_id": "CASE-R3-001",
            "action_type": "INITIAL_REVIEW",
            "action_at": "2026-03-03 12:00:00",
            "status": "COMPLETED"
        },
        {
            "action_id": "INV-002",
            "entity_id": "ORG-002",
            "case_id": "CASE-R3-001",
            "action_type": "DOCUMENT_REQUEST",
            "action_at": "2026-03-03 13:00:00",
            "status": "COMPLETED"
        }
    ])

    # -----------------------------------------------------
    # TELEMETRY
    # -----------------------------------------------------

    telemetry = pd.DataFrame([
        {
            "telemetry_id": "TEL-001",
            "entity_id": "ORG-001",
            "asset_id": "ASSET-001",
            "observed_at": "2026-03-01 10:00:00",
            "metric": "availability",
            "value": 99.9
        },
        {
            "telemetry_id": "TEL-002",
            "entity_id": "ORG-002",
            "asset_id": "ASSET-002",
            "observed_at": "2026-03-03 11:00:00",
            "metric": "availability",
            "value": 98.7
        }
    ])

    # -----------------------------------------------------
    # SAVE
    # -----------------------------------------------------

    escalations.to_csv(
        OUTPUT_DIR / "escalations.csv",
        index=False
    )

    investigation_actions.to_csv(
        OUTPUT_DIR / "investigation_actions.csv",
        index=False
    )

    telemetry.to_csv(
        OUTPUT_DIR / "telemetry.csv",
        index=False
    )

    print("Additional demo evidence created.")


if __name__ == "__main__":
    create_demo_evidence()

In [ ]:
from synthetic_data.demo.evidence import create_demo_evidence

create_demo_evidence()

In [ ]:
import pandas as pd

for filename in [
    "escalations.csv",
    "investigation_actions.csv",
    "telemetry.csv"
]:

    path = f"synthetic_data/demo/{filename}"

    df = pd.read_csv(path)

    print(
        f"{filename}: {len(df)} records"
    )

    display(df)

In [ ]:
from synthetic_data.demo_scenario import create_demo_scenario
from synthetic_data.demo.evidence import create_demo_evidence

create_demo_scenario()
create_demo_evidence()

In [ ]:
!python demo.py

In [ ]:
!python self_test.py

In [ ]:
!python -m py_compile \
    app.py \
    demo.py \
    analytics/pipeline.py

In [ ]:
!python -m py_compile \
    app.py \
    demo.py \
    analytics/pipeline.py

In [ ]:
!python -m py_compile \
    app.py \
    demo.py \
    analytics/pipeline.py

In [ ]:
!python -m py_compile \
    app.py \
    demo.py \
    analytics/pipeline.py

In [ ]:
import importlib
import sys

# Remove the module from sys.modules to force a fresh re-import
if 'ingestion.import_service' in sys.modules:
    del sys.modules['ingestion.import_service']

import ingestion.import_service

# Reload the module to pick up the latest schema definitions (redundant but harmless after del, for clarity)
importlib.reload(ingestion.import_service)

# Diagnostic: Print the schemas dictionary to confirm its contents
print("Current schemas in ingestion.import_service:")
print(ingestion.import_service.schemas.keys())

datasets = {
    "Cases":
        "synthetic_data/demo/cases.csv",

    "Alerts":
        "synthetic_data/demo/alerts.csv",

    "Assets":
        "synthetic_data/demo/assets.csv",

    "Escalations":
        "synthetic_data/demo/escalations.csv",

    "Investigation Actions":
        "synthetic_data/demo/investigation_actions.csv",

    "Telemetry":
        "synthetic_data/demo/telemetry.csv"
}


for dataset, path in datasets.items():

    df, result = ingestion.import_service.validate_uploaded_file(
        path,
        dataset
    )

    print(
        f"{dataset}: "
        f"{'PASS' if result['valid'] else 'FAIL'} "
        f"({result['row_count']} rows)"
    )